In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Load the text data
print("Loading dataset...")
df = pd.read_csv("../data/cleaned_arxiv_papers.csv") # Adjust path if needed based on your folder structure

# 2. Load the mathematical brain (Takes 1 second instead of 26 minutes!)
print("Loading embeddings...")
embeddings = np.load("../data/arxiv_embeddings.npy")

# 3. Load the model (We need this to convert the user's search query into a vector)
print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("System Ready! Embeddings shape:", embeddings.shape)

c:\Users\91749\Documents\CBInternship\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading dataset...
Loading embeddings...
Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2225.38it/s]


✅ System Ready! Embeddings shape: (15000, 384)


In [3]:
print(embeddings.shape)
print(embeddings.dtype)

(15000, 384)
float32




### **1. Why is the dtype `float32`? (The Decimal Point)**

When you look at `embedding.dtype`, it returns `float32`. This means every single number inside your massive array is a 32-bit floating-point number (a decimal number like `0.04561` or `-0.89213`).

* **Why decimals and not whole numbers?** Neural networks capture *subtle* semantic meanings. If they used integers (1, 2, 3), they couldn't capture fine details. A paper might be `0.78` similar to "AI" and `0.21` similar to "Biology". You need decimals to plot these exact coordinates in space.
* **Why `32-bit`?** In Deep Learning, `float32` is the golden industry standard. It gives the model plenty of mathematical precision to represent complex thoughts, but it takes up exactly half the computer memory as a `float64`. This is what allows your computer to calculate the Cosine Similarity so incredibly fast.

### **2. Why are there exactly 384 columns?**

Look at the shape: `(15000, 384)`.

* **15,000** is your number of rows (you successfully encoded 15,000 research papers).
* **384** is your number of columns. As we discussed in **Section 3 of your `dl_nlp_basics.md` notes**, this is the **dimensionality** of your embedding.
* **The "Why":** The specific neural network you downloaded (`all-MiniLM-L6-v2`) was hard-coded by its creators to output exactly 384 numbers. To the AI, every single one of those 384 columns represents a highly abstract "feature" of the text.
* Column 1 might measure how "academic" the text is.
* Column 2 might measure how strongly it relates to "algorithms".
* Column 3 might track "medical context".



*(Note: We can't actually read the columns like that as humans, but mathematically, that is how the neural network organizes the meaning).*

### **Interview Context:**

If an interviewer asks: *"Explain the output shape and type of your embedding matrix."*

**Your Answer:** *"My embedding matrix has a shape of 15,000 by 384, meaning I processed 15,000 documents, and the `MiniLM` model compressed the semantic meaning of each document into a 384-dimensional dense vector. The data type is `float32`, which is standard for neural network weights and embeddings because it perfectly balances computational speed, memory efficiency, and mathematical precision for downstream tasks like Cosine Similarity."*

In [4]:
import faiss

# 1. We must make a copy of the embeddings first because FAISS normalizes "in-place"
# (Meaning it permanently alters the original variable)
faiss_embeddings = embeddings.copy()

# 2. Apply L2 Normalization
faiss.normalize_L2(faiss_embeddings)

print("Embeddings normalized! Ready for the FAISS Index.")

Embeddings normalized! Ready for the FAISS Index.



### **FAISS L2 Normalization & Cosine Similarity**

**The Objective**
To normalize your 384-dimensional document embeddings (forcing their vector lengths to exactly 1) so that FAISS can use its lightning-fast Inner Product engine to calculate mathematically perfect Cosine Similarity scores.

**The Explanation**

* **Why are we normalizing?** In semantic search, the "meaning" of a sentence is stored in the *direction* the vector points, not how long the vector is. By scaling every single vector down to an exact length of 1, we remove any bias caused by vector size and force the math to look *only* at the angle.
* **Why specifically "L2"? (The Math Hack):** FAISS is incredibly fast but **does not** have a built-in Cosine Similarity function. However, it does have a blazing-fast Inner Product (Dot Product) function. The mathematical rule is: `L2 Normalization + Inner Product = Cosine Similarity`. By running `faiss.normalize_L2`, you are hacking FAISS to output the exact Cosine Similarity scores you want while keeping its extreme C++ speed.
* **The Output Range:** Because of this math hack, the scores FAISS will eventually return to you will fall into the standard Cosine Similarity range of **-1.0 to 1.0**:
* **1.0 (Maximum Score):** The vectors point in the exact same direction (identical semantic meaning).
* **0.0 (Neutral Score):** The vectors are perpendicular (unrelated meaning).
* **-1.0 (Minimum Score):** The vectors point in exact opposite directions.



*(Note: In real-world text data, you will almost never see negative scores. Most of your search results will score between `0.0` and `1.0`)*



In [7]:
index = faiss.IndexFlatIP(384)
index.add(faiss_embeddings)
index

<faiss.swigfaiss.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x00000171C1633BF0> >

### **FAISS Index Initialization**

**The Objective**
To create the empty structural container (the database "Index") in your computer's RAM, define the mathematical rules it will use to compare vectors, and tell it exactly what size those vectors will be.


**The Explanation**

* **What is `Index`?** In FAISS, an "Index" is essentially the database object. It is the highly optimized C++ data structure that will hold all your research paper embeddings in memory so they can be searched instantly.
* **What is `Flat`?** This tells FAISS how to search. "Flat" means it stores the vectors in a flat, uncompressed array and performs an **Exact Search** (brute-force). It will not skip any vectors; it will compare your search query to every single paper. *(Since you are only using a subset of about 15,000 papers, a Flat index is still incredibly fast. If you had 15 million papers, we would change this to `IVFFlat` to cluster the data).*
* **What is `IP`?** This stands for **Inner Product**. Because you just ran the L2 Normalization step in the previous cell, setting the index to `IP` is the final step of our "Math Hack." It tells the index to use the lightning-fast dot product multiplier, which will perfectly output Cosine Similarity scores.
* **What is `384`?** This is the dimensionality `(d)` of your vectors. FAISS needs to allocate the exact amount of memory required before it accepts any data. You are telling it: *"Get ready, I am about to hand you arrays that have exactly 384 floating-point numbers in them."*
* **Why is the index currently empty?** Right now, you just bought the bookshelf and assembled it, but you haven't put any books on it yet. You have defined the *rules* and the *size* of the database, but it currently holds `0` vectors.

The very next step is to actually take your `faiss_embeddings` matrix and "add" it to this empty index! Let me know when you are ready for that code.

### **Encoding the User Query**

**The Objective**
To take the raw text that a user types into the search bar and convert it into the exact same 384-dimensional mathematical language that our database of research papers uses.


In [ ]:
query = "deep learning for medical image analysis"
query_embedding = model.encode(query)
print(query_embedding.shape)

(1, 384)



**The Explanation**

* **`query = ...`**: This is simply simulating a user typing a search term into your final application.
* **`model.encode(query)`**: You are passing that raw text through the exact same `SentenceTransformer` (`all-MiniLM-L6-v2`) that you used to encode your 15,000 research papers. It is crucial to use the exact same model so that the query and the papers exist in the exact same mathematical space.
* **The Output `(384,)**`: This confirms the model successfully compressed your search phrase into a single, flat 1D array containing 384 floating-point numbers.

**Important Note for the Next Step:**
Remember the `ValueError` we hit way back when we first used `cosine_similarity`? Because FAISS (just like `sklearn`) expects to compare *matrices* (2D arrays), passing a flat `(384,)` array will cause an error. Before we search the FAISS index, we will need to quickly reshape this query vector into a 2D matrix of shape `(1, 384)`!

In [13]:
query_embedding = model.encode([query])
# query_embedding = model.encode(query).reshape(1,-1)
print(query_embedding.shape)
print(query_embedding)

(1, 384)
[[-3.52526270e-02  5.43267205e-02  5.65811619e-02 -4.97096591e-02
  -3.52174342e-02 -2.67348662e-02 -8.23495630e-03  1.52610037e-02
  -1.31554957e-02 -5.45781776e-02 -5.26165403e-02 -1.20118428e-02
  -8.10011551e-02  9.73842815e-02 -1.15813188e-01 -1.68417394e-02
  -9.74307582e-02 -5.52445743e-03 -1.06863678e-01  7.64726615e-03
  -4.89180721e-02 -6.45616092e-03 -3.65177169e-02  2.23113187e-02
   5.65221794e-02  4.34989482e-03  6.29342124e-02 -1.09063022e-01
   3.48554663e-02 -1.03746429e-02  7.24759325e-02 -3.63332741e-02
   1.78153962e-02  1.52815385e-02  4.68111448e-02  7.27785826e-02
  -1.72746703e-02  5.71633503e-02  6.73018489e-03  2.91875321e-02
   4.84562367e-02  5.15046231e-02  3.13239805e-02  5.97998910e-02
   8.63463804e-02 -1.21589913e-03  3.83907780e-02 -1.21435858e-02
   5.59613593e-02  9.04097706e-02 -1.46838073e-02  2.19262857e-02
  -7.75724947e-02  8.67380351e-02  3.64489332e-02 -1.94056972e-03
  -1.39828045e-02 -1.21372184e-02 -1.06874943e-01 -9.59569961e-03
 

In [14]:
faiss.normalize_L2(query_embedding)
query_embedding

array([[-3.52526270e-02,  5.43267205e-02,  5.65811619e-02,
        -4.97096591e-02, -3.52174342e-02, -2.67348662e-02,
        -8.23495630e-03,  1.52610037e-02, -1.31554957e-02,
        -5.45781776e-02, -5.26165403e-02, -1.20118428e-02,
        -8.10011551e-02,  9.73842815e-02, -1.15813188e-01,
        -1.68417394e-02, -9.74307582e-02, -5.52445743e-03,
        -1.06863678e-01,  7.64726615e-03, -4.89180721e-02,
        -6.45616092e-03, -3.65177169e-02,  2.23113187e-02,
         5.65221794e-02,  4.34989482e-03,  6.29342124e-02,
        -1.09063022e-01,  3.48554663e-02, -1.03746429e-02,
         7.24759325e-02, -3.63332741e-02,  1.78153962e-02,
         1.52815385e-02,  4.68111448e-02,  7.27785826e-02,
        -1.72746703e-02,  5.71633503e-02,  6.73018489e-03,
         2.91875321e-02,  4.84562367e-02,  5.15046231e-02,
         3.13239805e-02,  5.97998910e-02,  8.63463804e-02,
        -1.21589913e-03,  3.83907780e-02, -1.21435858e-02,
         5.59613593e-02,  9.04097706e-02, -1.46838073e-0

In [15]:
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

Scores (D): [[0.68072456 0.6709221  0.65219986 0.62811744 0.6131153 ]]
Indices (I): [[10466 13730 11873 12691 11282]]


This is the grand finale of your search engine backend, bro! You just successfully queried your FAISS database.

Here is the exact breakdown for your notes, including exactly what that `5` means!

### **Executing the FAISS Search**

**The Objective**
To search the FAISS index using the normalized user query, instantly finding the mathematical coordinates of the most relevant research papers and returning their similarity scores and row numbers.

**The Code**

```python
# Search the index!
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

```

**The Explanation**

* **What is the `5`? (Top-K):** In Machine Learning, this is known as `k`, which stands for how many results you want to retrieve. By passing `5`, you are telling FAISS: *"Find the top 5 closest vectors to my query."* If you wanted to show 10 results on a webpage, you would change this to `10`.
* **What is `D`? (Distances / Scores):** Look at your output: `[[0.6807...  0.6709... ]]`. Because we used our L2 Normalization + Inner Product hack, these are your **Cosine Similarity Scores**!
* The system found a paper with a `~0.68` match score.
* Notice how they are automatically sorted from highest to lowest. FAISS did all the heavy sorting logic for you!


* **What is `I`? (Indices / Row Numbers):** Look at your output: `[[10466 13730 ... ]]`. This is the most important part! FAISS is saying:
* *"The absolute best match is the paper sitting at **Row 10466** in your dataset."*
* *"The second best match is at **Row 13730**."*



### **Next Steps (The Final Piece of the Puzzle)**

Right now, you just have a row number (`10466`). A user doesn't care about row numbers; they want to read the actual title of the research paper!

Your very last step to complete this project is to take that `I` array, go back to your pandas DataFrame (`df`), and print out the title and abstract for row `10466`. Let me know if you want the quick `iloc` code to do that!